# BigAlpha 2026 — Hierarchical Transformer V5 Submission

本 Notebook 只负责平台推理：加载 `transformer_model.json`，并在平台注入的测试区间返回 `date / instrument / score`。训练、模型结构、数据构建和 JSON 序列化全部位于 `transformer_train.py`。

In [ ]:
"""BigAlpha 2026 V5 — load saved model and run inference only."""

import os
import structlog

from transformer_train import MODEL_PATH, predict, train_and_save

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """Platform entry: load trained JSON weights and return daily scores."""
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Missing {MODEL_PATH}; upload it with this notebook."
        )
    result = predict(datasources, start_date, end_date)
    logger.info(
        "score_ready",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    if not os.path.exists(MODEL_PATH):
        logger.info("model_missing_start_training", path=str(MODEL_PATH))
        train_and_save(datasources)
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
